In [2]:
import requests
import pandas as pd

# Direct MapServer endpoint captured from your browser network request
endpoint = "https://tnmap.tn.gov/arcgis/rest/services/SAFETY/MapForDashboards/MapServer/0/query"

# SQL condition for both Fatal and Suspected Serious Injury crashes
where_clause = "Crash_Type IN ('Fatal', 'Suspected Serious Injury')"

# Requested fields matching your network request payload

all_records = []
offset = 0
record_limit = 1000

while True:
    params = {
        "where": where_clause,
        "outFields": "*",
        "f": "json",  # Overrides 'pbf' so python can parse standard JSON easily
        "resultOffset": offset,
        "resultRecordCount": record_limit,
        "orderByFields": "OBJECTID ASC",
        "returnGeometry": "false"  # Set to "true" if you need spatial coordinates
    }

    response = requests.get(endpoint, params=params)
    data = response.json()
    
    features = data.get("features", [])
    if not features:
        break

    # Extract attribute dictionaries
    records = [f["attributes"] for f in features]
    all_records.extend(records)
    
    # Break loop if we've fetched all available records
    if len(features) < record_limit:
        break
        
    offset += record_limit
    print(f"Fetched {len(all_records)} records so far...")

# Convert to DataFrame
df = pd.DataFrame(all_records)

# Convert Epoch timestamps (milliseconds) in Collision_Date to human-readable dates
if "Collision_Date" in df.columns:
    df["Collision_Date"] = pd.to_datetime(df["Collision_Date"], unit="ms")

# Save output to CSV
df.to_csv("tn_fatal_and_serious_injury_crashes.csv", index=False)
print(f"Successfully saved {len(df)} total crash records to tn_fatal_and_serious_injury_crashes.csv")

Fetched 1000 records so far...
Fetched 2000 records so far...
Fetched 3000 records so far...
Fetched 4000 records so far...
Fetched 5000 records so far...
Fetched 6000 records so far...
Fetched 7000 records so far...
Fetched 8000 records so far...
Fetched 9000 records so far...
Fetched 10000 records so far...
Fetched 11000 records so far...
Fetched 12000 records so far...
Fetched 13000 records so far...
Fetched 14000 records so far...
Fetched 15000 records so far...
Successfully saved 15126 total crash records to tn_fatal_and_serious_injury_crashes.csv


In [6]:
import requests
import pandas as pd
pd.set_option('display.max_columns', None)
crash = pd.read_csv("tn_fatal_and_serious_injury_crashes.csv")

In [11]:
crash['Crash_Type'].unique()

<StringArray>
['Suspected Serious Injury', 'Fatal']
Length: 2, dtype: str

In [15]:
crash["OBJECTID"].unique()

array([    46,     61,     77, ..., 120539, 120540, 120569],
      shape=(15126,))

In [17]:
crash['Collision_Date'][0]

'2024-01-03 14:55:00'

In [26]:
crash['LargeTruck_Involved'].unique()

<StringArray>
['-', 'ATV_Involved']
Length: 2, dtype: str

In [27]:
crash['Injury'].unique()

array([ 5,  4,  1,  2,  7,  0,  3,  6,  8, 10, 11, 39, 29,  9, 13, 15])

In [28]:
crash['Fatality'].unique()

array([0, 1, 2, 4, 3, 6])

In [32]:
crash['Crash_Type'].unique()

<StringArray>
['Suspected Serious Injury', 'Fatal']
Length: 2, dtype: str